In [ ]:
# --- Repository setup ---
import os
from pathlib import Path

# Find the repository root (the folder that contains data/graphs) and work from there,
# so every relative path below resolves no matter where Jupyter was started.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'graphs').is_dir())
os.chdir(REPO_ROOT)
print('REPO_ROOT:', REPO_ROOT)


# Phase 1 — Simple AI-Assisted Graph to JSON Generation
This notebook intentionally keeps the implementation simple.

The project already has module graphs. The notebook takes those graph objects as input and asks an AI model to generate the required structured JSON output.

There are only two main flows:
- `generate_single_hop_json(graph)` for one module graph
- `generate_multi_hop_json(source_graph, target_graph)` for two related module graphs

The traversal and classification complexity is delegated to the AI prompt. The Python code focuses on loading graphs, calling the model, validating shape, saving JSON, and testing the integration path.


## Configuration and imports
Set `PROJECT_ROOT` if your graph folder is somewhere else. Set `OPENAI_API_KEY` in your environment before running real AI generation.

Tests in this notebook do **not** require an API key because they use a fake AI generator.


In [9]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any, Callable

# Optional if your notebook environment does not already have the SDK:
# %pip install -q openai

PROJECT_ROOT = REPO_ROOT / 'data' / 'graphs'
OUTPUT_ROOT = REPO_ROOT / 'data' / 'cases' / 'reference_paths'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Strong default model. Change to 'gpt-5.5' if you want lower cost/latency.
OPENAI_MODEL = 'gpt-5.5-pro'

print('Project root:', PROJECT_ROOT)
print('Output root :', OUTPUT_ROOT)
print('Model       :', OPENAI_MODEL)


Project root: /Users/rakib/Downloads/mental-health-research/mental-health-research
Output root : /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple
Model       : gpt-5.5-pro


## Small graph I/O helpers
These helpers only load, discover, and save JSON. They do not try to interpret the clinical graph.


In [10]:
def load_graph(path: str | Path) -> dict[str, Any]:
    path = Path(path)
    graph = json.loads(path.read_text(encoding='utf-8'))
    validate_graph_shape(graph, graph_name=str(path))
    return graph


def discover_graph_files(root: str | Path = PROJECT_ROOT) -> list[Path]:
    return sorted(Path(root).rglob('mhgap_decision_graph.json'))


def save_json(data: Any, path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding='utf-8')
    return path


def validate_graph_shape(graph: dict[str, Any], graph_name: str = 'graph') -> None:
    required = {'module', 'nodes', 'relationships'}
    missing = required - set(graph)
    if missing:
        raise ValueError(f'{graph_name} is missing required keys: {sorted(missing)}')
    if not isinstance(graph['nodes'], list):
        raise ValueError(f'{graph_name}.nodes must be a list')
    if not isinstance(graph['relationships'], list):
        raise ValueError(f'{graph_name}.relationships must be a list')


def module_id(graph: dict[str, Any]) -> str:
    return str(graph.get('module', {}).get('id', 'UNKNOWN'))


## AI helper
All OpenAI-specific code lives in one function. This keeps the rest of the notebook easy to test and maintain.

The helper returns parsed JSON with a single top-level key:
- `cases`: list of generated cases

No `warnings`, `metadata`, or `decision_trace` fields are requested or saved.


In [11]:
AI_OUTPUT_JSON_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'required': ['cases'],
    'properties': {
        'cases': {
            'type': 'array',
            'items': {
                'type': 'object',
                'additionalProperties': True,
                'required': [
                    'case_id',
                    'category',
                    'source_module',
                    'referred_module',
                    'final_module',
                    'diagnosis_reached',
                    'diagnosis_node_id',
                    'exit_node_id',
                    'node_sequence',
                ],
                'properties': {
                    'case_id': {'type': 'string'},
                    'category': {
                        'type': 'string',
                        'enum': [
                            'single_hop_positive',
                            'single_hop_negative',
                            'multi_hop_positive',
                            'multi_hop_negative',
                        ],
                    },
                    'source_module': {'type': 'string'},
                    'referred_module': {'anyOf': [{'type': 'string'}, {'type': 'null'}]},
                    'final_module': {'type': 'string'},
                    'diagnosis_reached': {'type': 'boolean'},
                    'diagnosis_node_id': {'anyOf': [{'type': 'string'}, {'type': 'null'}]},
                    'exit_node_id': {'anyOf': [{'type': 'string'}, {'type': 'null'}]},
                    'node_sequence': {'type': 'array', 'items': {'type': 'object', 'additionalProperties': True}},
                },
            },
        },
    },
}


def extract_json_object(text: str) -> dict[str, Any]:
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find('{')
        end = text.rfind('}')
        if start == -1 or end == -1 or end <= start:
            raise ValueError('AI response did not contain a JSON object')
        return json.loads(text[start:end + 1])


def call_openai_json(task: str, payload: dict[str, Any], model: str = OPENAI_MODEL) -> dict[str, Any]:
    if not os.getenv('OPENAI_API_KEY'):
        raise RuntimeError('OPENAI_API_KEY is not set. Set it before running real AI generation.')

    try:
        from openai import OpenAI
    except ModuleNotFoundError as exc:
        raise RuntimeError('OpenAI SDK is not installed. In a notebook cell, run: %pip install openai') from exc

    client = OpenAI()
    request = {
        'model': model,
        'max_output_tokens': 20000,
        'text': {
            'format': {
                'type': 'json_schema',
                'name': 'graph_generation_result',
                'schema': AI_OUTPUT_JSON_SCHEMA,
                'strict': False,
            }
        },
        'input': [
            {
                'role': 'developer',
                'content': (
                    'You convert mental-health decision graph JSON into the requested output JSON. '
                    'Return one valid JSON object matching the provided schema. Do not include markdown. '
                    'The top-level object must contain only cases. Do not include warnings, metadata, or decision_trace. '
                    'Do not invent graph nodes or relationships. If graph data is incomplete or no supported new case exists, return {"cases": []}.'
                ),
            },
            {
                'role': 'user',
                'content': json.dumps({'task': task, 'payload': payload}, ensure_ascii=False),
            },
        ],
    }

    try:
        response = client.responses.create(**request)
    except TypeError:
        request.pop('text', None)
        response = client.responses.create(**request)

    raw_text = response.output_text or ''
    debug_path = OUTPUT_ROOT / 'last_openai_raw_response.json.txt'
    debug_path.write_text(raw_text, encoding='utf-8')

    try:
        return extract_json_object(raw_text)
    except json.JSONDecodeError as exc:
        raise ValueError(
            f'OpenAI returned malformed JSON. Raw response saved to {debug_path}. '
            f'Original parse error: {exc}'
        ) from exc


## Output schema validation
This is intentionally lightweight. It checks that the AI returned only `cases` and that each case contains the required fields.


In [12]:
CASE_REQUIRED_KEYS = {
    'case_id',
    'category',
    'source_module',
    'referred_module',
    'final_module',
    'diagnosis_reached',
    'diagnosis_node_id',
    'exit_node_id',
    'node_sequence',
}

VALID_CATEGORIES = {
    'single_hop_positive',
    'single_hop_negative',
    'multi_hop_positive',
    'multi_hop_negative',
}


def validate_ai_output(result: dict[str, Any]) -> None:
    if not isinstance(result, dict):
        raise ValueError('AI result must be a JSON object')
    extra_top_level_keys = set(result) - {'cases'}
    if extra_top_level_keys:
        raise ValueError(f'AI result should only contain cases; unexpected keys: {sorted(extra_top_level_keys)}')
    if 'cases' not in result:
        raise ValueError('AI result must contain a cases list')
    if not isinstance(result['cases'], list):
        raise ValueError('AI result cases must be a list')

    for index, case in enumerate(result['cases']):
        missing = CASE_REQUIRED_KEYS - set(case)
        if missing:
            raise ValueError(f'Case {index} is missing keys: {sorted(missing)}')
        if 'decision_trace' in case:
            raise ValueError(f'Case {index} should not include decision_trace')
        if case['category'] not in VALID_CATEGORIES:
            raise ValueError(f"Case {index} has invalid category: {case['category']}")
        if not isinstance(case['node_sequence'], list):
            raise ValueError(f'Case {index}.node_sequence must be a list')


## Single-hop generation
Input: one module graph.

Each OpenAI call requests **at most one case**. Use `case_count` to request several cases; the helper loops one case at a time and stops early when the model says no supported case exists.

You can also target a specific category, for example `requested_category='single_hop_positive'`.


In [13]:
def summarize_case_for_prompt(case: dict[str, Any]) -> dict[str, Any]:
    node_sequence = case.get('node_sequence', [])
    if not isinstance(node_sequence, list):
        node_sequence = []

    relationship_sequence = case.get('relationship_sequence', [])
    if not isinstance(relationship_sequence, list):
        relationship_sequence = []

    return {
        'case_id': case.get('case_id'),
        'category': case.get('category'),
        'node_ids': [node.get('id') for node in node_sequence],
        # New: flow is inline on nodes. Each element corresponds to the condition used
        # to reach the next node in the sequence.
        'step_conditions': [node.get('condition_met') for node in node_sequence[:-1]],
        # Back-compat: if old cases still include relationship_sequence, keep a compact form.
        'relationship_ids': [
            [rel.get('from'), rel.get('condition'), rel.get('to')]
            for rel in relationship_sequence
        ],
    }


def merge_generation_results(results: list[dict[str, Any]]) -> dict[str, Any]:
    def signature_for_case(case: dict[str, Any]) -> str:
        node_sequence = case.get('node_sequence', [])
        if not isinstance(node_sequence, list):
            node_sequence = []

        node_ids = [node.get('id') for node in node_sequence]
        step_conditions = [node.get('condition_met') for node in node_sequence[:-1]]

        # If condition_met wasn't provided, fall back to relationship_sequence when possible.
        if node_ids and (not step_conditions or all(c is None for c in step_conditions)):
            relationship_sequence = case.get('relationship_sequence', [])
            if isinstance(relationship_sequence, list) and len(relationship_sequence) == max(0, len(node_ids) - 1):
                step_conditions = [rel.get('condition') for rel in relationship_sequence]

        return json.dumps(
            {
                'category': case.get('category'),
                'node_ids': node_ids,
                'step_conditions': step_conditions,
            },
            sort_keys=True,
            ensure_ascii=False,
        )

    cases: list[dict[str, Any]] = []
    seen_signatures: set[str] = set()

    for result in results:
        for case in result.get('cases', []):
            signature = signature_for_case(case)
            if signature in seen_signatures:
                continue
            seen_signatures.add(signature)
            cases.append(case)

    return {'cases': cases}


def single_hop_prompt_payload(
    graph: dict[str, Any],
    *,
    requested_category: str | None = None,
    existing_cases: list[dict[str, Any]] | None = None,
) -> dict[str, Any]:
    validate_graph_shape(graph)
    if requested_category and requested_category not in {'single_hop_positive', 'single_hop_negative'}:
        raise ValueError('requested_category must be single_hop_positive, single_hop_negative, or None')

    return {
        'flow_type': 'single_hop',
        'requested_category': requested_category,
        'max_cases_to_return': 1,
        'existing_cases_to_avoid': [
            summarize_case_for_prompt(case) for case in (existing_cases or [])
        ],
        'instructions': [
            'Use only the provided module graph.',
            'Generate at most one case for this request.',
            'If requested_category is set, generate only that category.',
            'If no supported new case exists for requested_category, return exactly {"cases": []}.',
            'Start from the node where is_starting is true.',
            'Classify the case as single_hop_positive or single_hop_negative.',
            'Positive means the path reaches a diagnosis confirming the module syndrome.',
            'Negative means the path exits or rules out the module syndrome without positive diagnosis.',
            'Avoid duplicating any path listed in existing_cases_to_avoid.',
            'Preserve node objects from graph.nodes. Do not invent or rewrite node text.',
            'Inline the flow in node_sequence: for each node except the last, add condition_met equal to the relationship.condition used to reach the next node in the sequence.',
            'condition_met must match an edge in graph.relationships with from=this_node.id and to=next_node.id.',
            'Do not return relationship_sequence.',
            'Do not include decision_trace, warnings, or metadata.',
            'Do not invent missing nodes, missing edges, or unsupported paths.',
            'Do not create unrealistic data to satisfy a requested count.',
        ],
        'required_case_schema': list(CASE_REQUIRED_KEYS),
        'graph': graph,
    }


def generate_single_hop_case(
    graph: dict[str, Any],
    *,
    requested_category: str | None = None,
    existing_cases: list[dict[str, Any]] | None = None,
    ai_generator: Callable[[str, dict[str, Any]], dict[str, Any]] = call_openai_json,
) -> dict[str, Any]:
    payload = single_hop_prompt_payload(
        graph,
        requested_category=requested_category,
        existing_cases=existing_cases,
    )
    result = ai_generator('single_hop_one_case_to_json', payload)
    validate_ai_output(result)
    if len(result['cases']) > 1:
        result['cases'] = result['cases'][:1]
    return result


def generate_single_hop_json(
    graph: dict[str, Any],
    *,
    requested_category: str | None = None,
    case_count: int = 1,
    ai_generator: Callable[[str, dict[str, Any]], dict[str, Any]] = call_openai_json,
) -> dict[str, Any]:
    if case_count < 1:
        raise ValueError('case_count must be >= 1')

    results: list[dict[str, Any]] = []
    generated_cases: list[dict[str, Any]] = []

    for _ in range(case_count):
        result = generate_single_hop_case(
            graph,
            requested_category=requested_category,
            existing_cases=generated_cases,
            ai_generator=ai_generator,
        )
        if not result['cases']:
            break
        results.append(result)
        generated_cases.extend(result['cases'])

    return merge_generation_results(results)


## Multi-hop generation
Input: two module graphs.

Each OpenAI call requests **at most one case** for the source → target relationship. Use `case_count` to request several cases, and the helper stops early rather than inventing unrealistic paths.

You can target `multi_hop_positive` or `multi_hop_negative` with `requested_category`.


In [14]:
def multi_hop_prompt_payload(
    source_graph: dict[str, Any],
    target_graph: dict[str, Any],
    *,
    requested_category: str | None = None,
    existing_cases: list[dict[str, Any]] | None = None,
) -> dict[str, Any]:
    validate_graph_shape(source_graph, 'source_graph')
    validate_graph_shape(target_graph, 'target_graph')
    if requested_category and requested_category not in {'multi_hop_positive'}:
        raise ValueError('requested_category must be multi_hop_positive, multi_hop_negative, or None')

    return {
        'flow_type': 'multi_hop',
        'requested_category': requested_category,
        'max_cases_to_return': 1,
        'existing_cases_to_avoid': [
            summarize_case_for_prompt(case) for case in (existing_cases or [])
        ],
        'instructions': [
            'Use exactly two graphs: source_graph and target_graph.',
            'Generate at most one case for this request.',
            'If requested_category is set, generate only that category.',
            'If no supported new case exists for requested_category, return exactly {"cases": []}.',
            'Find referral nodes in source_graph whose target_module matches target_graph.module.id.',
            'A multi-hop case starts at the source starting node, reaches one supported referral, then continues from the target graph starting node.',
            'Allow only one referral hop total.',
            'If a second referral is needed or encountered, skip that unsupported case.',
            'Classify the case as multi_hop_positive or multi_hop_negative.',
            'Positive means the target graph path reaches a diagnosis confirming the referred module syndrome.',
            'Negative means the target graph path exits or rules out the referred module syndrome without positive diagnosis.',
            'Avoid duplicating any path listed in existing_cases_to_avoid.',
            'Preserve node objects from the provided graph nodes. Do not invent or rewrite node text.',
            'Inline the flow in node_sequence: for each node except the last, add condition_met equal to the relationship.condition used to reach the next node in the sequence (within the same graph).',
            'When the sequence crosses from the source graph to the target graph (the referral hop), omit condition_met on the referral node.',
            'Do not return relationship_sequence.',
            'Do not include decision_trace, warnings, or metadata.',
            'Do not invent missing nodes, missing edges, or unsupported paths.',
            'Do not create unrealistic data to satisfy a requested count.',
        ],
        'required_case_schema': list(CASE_REQUIRED_KEYS),
        'source_graph': source_graph,
        'target_graph': target_graph,
    }


def generate_multi_hop_case(
    source_graph: dict[str, Any],
    target_graph: dict[str, Any],
    *,
    requested_category: str | None = None,
    existing_cases: list[dict[str, Any]] | None = None,
    ai_generator: Callable[[str, dict[str, Any]], dict[str, Any]] = call_openai_json,
) -> dict[str, Any]:
    payload = multi_hop_prompt_payload(
        source_graph,
        target_graph,
        requested_category=requested_category,
        existing_cases=existing_cases,
    )
    result = ai_generator('multi_hop_one_case_to_json', payload)
    validate_ai_output(result)
    if len(result['cases']) > 1:
        result['cases'] = result['cases'][:1]
    return result


def generate_multi_hop_json(
    source_graph: dict[str, Any],
    target_graph: dict[str, Any],
    *,
    requested_category: str | None = None,
    case_count: int = 1,
    ai_generator: Callable[[str, dict[str, Any]], dict[str, Any]] = call_openai_json,
) -> dict[str, Any]:
    if case_count < 1:
        raise ValueError('case_count must be >= 1')

    results: list[dict[str, Any]] = []
    generated_cases: list[dict[str, Any]] = []

    for _ in range(case_count):
        result = generate_multi_hop_case(
            source_graph,
            target_graph,
            requested_category=requested_category,
            existing_cases=generated_cases,
            ai_generator=ai_generator,
        )
        if not result['cases']:
            break
        results.append(result)
        generated_cases.extend(result['cases'])

    return merge_generation_results(results)


## Convenience runners
Use these helpers for real project graph files. They call OpenAI unless you pass another `ai_generator`.


In [15]:
def run_single_hop_file(
    path: str | Path,
    *,
    requested_category: str | None = None,
    case_count: int = 1,
    ai_generator: Callable[[str, dict[str, Any]], dict[str, Any]] = call_openai_json,
) -> Path:
    graph = load_graph(path)
    result = generate_single_hop_json(
        graph,
        requested_category=requested_category,
        case_count=case_count,
        ai_generator=ai_generator,
    )
    category_label = requested_category or 'any'
    module_folder = OUTPUT_ROOT / module_id(graph)
    out_path = module_folder / f'{module_id(graph)}_single_hop_{category_label}_{len(result["cases"])}.json'
    return save_json(result, out_path)


def run_multi_hop_files(
    source_path: str | Path,
    target_path: str | Path,
    *,
    requested_category: str | None = None,
    case_count: int = 1,
    ai_generator: Callable[[str, dict[str, Any]], dict[str, Any]] = call_openai_json,
) -> Path:
    source_graph = load_graph(source_path)
    target_graph = load_graph(target_path)
    result = generate_multi_hop_json(
        source_graph,
        target_graph,
        requested_category=requested_category,
        case_count=case_count,
        ai_generator=ai_generator,
    )
    category_label = requested_category or 'any'
    module_folder = OUTPUT_ROOT / module_id(source_graph)
    out_path = module_folder / f'{module_id(source_graph)}_to_{module_id(target_graph)}_multi_hop_{category_label}_{len(result["cases"])}.json'
    return save_json(result, out_path)


def show_available_graphs() -> list[Path]:
    paths = discover_graph_files(PROJECT_ROOT)
    for path in paths:
        print(path)
    return paths


## Example usage with real files
Uncomment and run these cells when `OPENAI_API_KEY` is set.

The first line lists graph files. Pick paths from that output for single-hop or multi-hop generation.


In [16]:
graph_paths = show_available_graphs()


/Users/rakib/Downloads/mental-health-research/mental-health-research/CHILD & ADOLESCENT MENTAL & BEHAVIOURAL DISORDERS/mhgap_decision_graph.json
/Users/rakib/Downloads/mental-health-research/mental-health-research/DEMENTIA/mhgap_decision_graph.json
/Users/rakib/Downloads/mental-health-research/mental-health-research/DEPRESSION/Extra files/mhgap_decision_graph.json
/Users/rakib/Downloads/mental-health-research/mental-health-research/DEPRESSION/mhgap_decision_graph.json
/Users/rakib/Downloads/mental-health-research/mental-health-research/DISORDERS DUE TO SUBSTANCE USE/mhgap_decision_graph.json
/Users/rakib/Downloads/mental-health-research/mental-health-research/EPILEPSY/mhgap_decision_graph.json
/Users/rakib/Downloads/mental-health-research/mental-health-research/OTHER SIGNIFICANT MENTAL HEALTH COMPLAINTS/mhgap_decision_graph.json
/Users/rakib/Downloads/mental-health-research/mental-health-research/PSYCHOSES/mhgap_decision_graph.json
/Users/rakib/Downloads/mental-health-research/mental-h

## Batch generation by category
Each category is split into three cells: single-hop positive, single-hop negative, then multi-hop loops.


In [17]:
# valid_paths = [
#     ['DEP', 'SUI'], ['DEP', 'CMH'], ['DEP', 'PSY'], ['DEP', 'OTH'],
#     ['CMH', 'SUI'], ['CMH', 'SUB'], ['CMH', 'EPI'], ['CMH', 'DEP'],
#     ['SUB', 'SUI'], ['SUB', 'PSY'], ['SUB', 'CMH'],
#     ['EPI', 'SUI'],
#     ['OTH', 'CMH'], ['OTH', 'SUI'],
#     ['PSY', 'SUI'], ['PSY', 'SUB'], ['PSY', 'DEP'],
#     ['SUI', 'DEP'], ['SUI', 'SUB'], ['SUI', 'EPI'], ['SUI', 'OTH'], ['SUI', 'PSY'],
# ]

valid_paths = [
    ['DEP', 'CMH'], ['DEP', 'PSY'], ['DEP', 'OTH'],
    ['CMH', 'SUI'], ['CMH', 'SUB'], ['CMH', 'EPI'], ['CMH', 'DEP'],
    ['SUB', 'SUI'], ['SUB', 'PSY'], ['SUB', 'CMH'],
    ['EPI', 'SUI'],
    ['OTH', 'CMH'], ['OTH', 'SUI'],
    ['PSY', 'SUI'], ['PSY', 'SUB'], ['PSY', 'DEP'],
    ['SUI', 'DEP'], ['SUI', 'SUB'], ['SUI', 'EPI'], ['SUI', 'OTH'], ['SUI', 'PSY'],
]


MODULE_DIRS = {
    'DEP': 'DEP',
    'CMH': 'CMH',
    'SUB': 'SUB',
    'EPI': 'EPI',
    'OTH': 'OTH',
    'PSY': 'PSY',
    'SUI': 'SUI',
}

SINGLE_HOP_POS_COUNT = 1
SINGLE_HOP_NEG_COUNT = 1
MULTI_HOP_COUNT = 1

def graph_path(module_code: str) -> Path:
    return PROJECT_ROOT / MODULE_DIRS[module_code] / 'mhgap_decision_graph.json'

def iter_multi_hop_from(start_code: str):
    for src, dst in valid_paths:
        if src == start_code:
            yield src, dst


### DEP


In [10]:
out = run_single_hop_file(
    graph_path('DEP'),
    requested_category='single_hop_positive',
    case_count=SINGLE_HOP_POS_COUNT,
)
print('Saved:', out)


Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/DEP/DEP_single_hop_single_hop_positive_1.json


In [14]:
out = run_single_hop_file(
    graph_path('DEP'),
    requested_category='single_hop_negative',
    case_count=SINGLE_HOP_NEG_COUNT,
)
print('Saved:', out)


Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/DEP/DEP_single_hop_single_hop_negative_1.json


In [18]:


for src, dst in iter_multi_hop_from('DEP'):
        out = run_multi_hop_files(
            graph_path(src),
            graph_path(dst),
            requested_category='multi_hop_positive',
            case_count=MULTI_HOP_COUNT,
        )
        print(f'{src}->{dst} multi_hop_positive Saved:', out)



DEP->CMH multi_hop_positive Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/DEP/DEP_to_CMH_multi_hop_multi_hop_positive_1.json
DEP->PSY multi_hop_positive Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/DEP/DEP_to_PSY_multi_hop_multi_hop_positive_1.json
DEP->OTH multi_hop_positive Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/DEP/DEP_to_OTH_multi_hop_multi_hop_positive_1.json


### CMH


In [19]:
out = run_single_hop_file(
    graph_path('CMH'),
    requested_category='single_hop_positive',
    case_count=SINGLE_HOP_POS_COUNT,
)
print('Saved:', out)


Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/CMH/CMH_single_hop_single_hop_positive_1.json


In [18]:
out = run_single_hop_file(
    graph_path('CMH'),
    requested_category='single_hop_negative',
    case_count=SINGLE_HOP_NEG_COUNT,
)
print('Saved:', out)


Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/CMH/CMH_single_hop_single_hop_negative_1.json


In [22]:
# for src, dst in iter_multi_hop_from('CMH'):
#         out = run_multi_hop_files(
#             graph_path(src),
#             graph_path(dst),
#             requested_category='multi_hop_positive',
#             case_count=MULTI_HOP_COUNT,
#         )
#         print(f'{src}->{dst} multi_hop_positive Saved:', out)




out = run_multi_hop_files(
            graph_path('CMH'),
            graph_path('DEP'),
            requested_category='multi_hop_positive',
            case_count=MULTI_HOP_COUNT,
        )

### SUB


In [26]:
out = run_single_hop_file(
    graph_path('SUB'),
    requested_category='single_hop_positive',
    case_count=SINGLE_HOP_POS_COUNT,
)
print('Saved:', out)


Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/SUB/SUB_single_hop_single_hop_positive_1.json


In [27]:
out = run_single_hop_file(
    graph_path('SUB'),
    requested_category='single_hop_negative',
    case_count=SINGLE_HOP_NEG_COUNT,
)
print('Saved:', out)


Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/SUB/SUB_single_hop_single_hop_negative_1.json


In [28]:
for src, dst in iter_multi_hop_from('SUB'):
        out = run_multi_hop_files(
            graph_path(src),
            graph_path(dst),
            requested_category='multi_hop_positive',
            case_count=MULTI_HOP_COUNT,
        )
        print(f'{src}->{dst} multi_hop_positive Saved:', out)



SUB->SUI multi_hop_positive Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/SUB/SUB_to_SUI_multi_hop_multi_hop_positive_1.json
SUB->PSY multi_hop_positive Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/SUB/SUB_to_PSY_multi_hop_multi_hop_positive_1.json
SUB->CMH multi_hop_positive Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/SUB/SUB_to_CMH_multi_hop_multi_hop_positive_0.json


### EPI


In [23]:
out = run_single_hop_file(
    graph_path('EPI'),
    requested_category='single_hop_positive',
    case_count=SINGLE_HOP_POS_COUNT,
)
print('Saved:', out)


Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/EPI/EPI_single_hop_single_hop_positive_1.json


In [24]:
out = run_single_hop_file(
    graph_path('EPI'),
    requested_category='single_hop_negative',
    case_count=SINGLE_HOP_NEG_COUNT,
)
print('Saved:', out)


Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/EPI/EPI_single_hop_single_hop_negative_1.json


In [ ]:

# for src, dst in iter_multi_hop_from('EPI'):
#         out = run_multi_hop_files(
#             graph_path(src),
#             graph_path(dst),
#             requested_category='multi_hop_positive',
#             case_count=MULTI_HOP_COUNT,
#         )
#         print(f'{src}->{dst} multi_hop_positive Saved:', out)

out = run_multi_hop_files(
            graph_path(src),
            graph_path(dst),
            requested_category='multi_hop_positive',
            case_count=MULTI_HOP_COUNT,
        )
print(f'{src}->{dst} multi_hop_positive Saved:', out)


EPI->SUI multi_hop_positive Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/EPI/EPI_to_SUI_multi_hop_multi_hop_positive_1.json


### OTH


In [ ]:
out = run_single_hop_file(
    graph_path('OTH'),
    requested_category='single_hop_positive',
    case_count=SINGLE_HOP_POS_COUNT,
)
print('Saved:', out)


In [ ]:
out = run_single_hop_file(
    graph_path('OTH'),
    requested_category='single_hop_negative',
    case_count=SINGLE_HOP_NEG_COUNT,
)
print('Saved:', out)


In [ ]:
for src, dst in iter_multi_hop_from('OTH'):
        out = run_multi_hop_files(
            graph_path(src),
            graph_path(dst),
            requested_category='multi_hop_positive',
            case_count=MULTI_HOP_COUNT,
        )
        print(f'{src}->{dst} multi_hop_positive Saved:', out)



### PSY


In [29]:
out = run_single_hop_file(
    graph_path('PSY'),
    requested_category='single_hop_positive',
    case_count=SINGLE_HOP_POS_COUNT,
)
print('Saved:', out)


Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/PSY/PSY_single_hop_single_hop_positive_1.json


In [30]:
out = run_single_hop_file(
    graph_path('PSY'),
    requested_category='single_hop_negative',
    case_count=SINGLE_HOP_NEG_COUNT,
)
print('Saved:', out)


Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/PSY/PSY_single_hop_single_hop_negative_1.json


In [31]:
for src, dst in iter_multi_hop_from('PSY'):
        out = run_multi_hop_files(
            graph_path(src),
            graph_path(dst),
            requested_category='multi_hop_positive',
            case_count=MULTI_HOP_COUNT,
        )
        print(f'{src}->{dst} multi_hop_positive Saved:', out)



PSY->SUI multi_hop_positive Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/PSY/PSY_to_SUI_multi_hop_multi_hop_positive_1.json
PSY->SUB multi_hop_positive Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/PSY/PSY_to_SUB_multi_hop_multi_hop_positive_0.json
PSY->DEP multi_hop_positive Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/PSY/PSY_to_DEP_multi_hop_multi_hop_positive_0.json


### SUI


In [32]:
out = run_single_hop_file(
    graph_path('SUI'),
    requested_category='single_hop_positive',
    case_count=SINGLE_HOP_POS_COUNT,
)
print('Saved:', out)


Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/SUI/SUI_single_hop_single_hop_positive_1.json


In [33]:
out = run_single_hop_file(
    graph_path('SUI'),
    requested_category='single_hop_negative',
    case_count=SINGLE_HOP_NEG_COUNT,
)
print('Saved:', out)


Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/SUI/SUI_single_hop_single_hop_negative_1.json


In [34]:
for src, dst in iter_multi_hop_from('SUI'):
        out = run_multi_hop_files(
            graph_path(src),
            graph_path(dst),
            requested_category='multi_hop_positive',
            case_count=MULTI_HOP_COUNT,
        )
        print(f'{src}->{dst} multi_hop_positive Saved:', out)



SUI->DEP multi_hop_positive Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/SUI/SUI_to_DEP_multi_hop_multi_hop_positive_0.json
SUI->SUB multi_hop_positive Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/SUI/SUI_to_SUB_multi_hop_multi_hop_positive_0.json
SUI->EPI multi_hop_positive Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/SUI/SUI_to_EPI_multi_hop_multi_hop_positive_0.json
SUI->OTH multi_hop_positive Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/SUI/SUI_to_OTH_multi_hop_multi_hop_positive_1.json
SUI->PSY multi_hop_positive Saved: /Users/rakib/Downloads/mental-health-research/detector_outputs/phase1_ai_simple/SUI/SUI_to_PSY_multi_hop_multi_hop_positive_0.json
